# Notebook 20 — Positional-only & keyword-only APIs (`/` `*`)

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `19-httpx-http-clients.ipynb`

**Next up:** `21-venv-and-dependency-pins.ipynb`

---

## Learning objectives

- Freeze brittle positional arguments (`prompt`, `messages`).
- Force keyword clarity for knobs (`temperature`, `stream`).
- Design wrappers that survive signature churn across vendors.

## Table of contents

1. **`/` positional-only zone**
2. **`*` keyword-only zone**
3. **Combined staging shapes**
4. **Progressive drills — clarify calls → forbid drift → orchestrator façade**
5. **Exercise — `tool_call` boundary**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · Positional-only (`/`)

*Explanation:* **`/`** stops callers from passing prompts positionally beyond what you intend — protects wrappers embedding **`messages`** arrays.


In [ ]:
def relay(prompt: str, /, vendor: str) -> str:
    return f"{vendor}:{prompt[:12]}"


print(relay("Summarize outage timeline", vendor="east"))


## 2 · Keyword-only (`*`)

*Explanation:* After **`*`**, callers **must** name **`temperature=`**, **`stream=`**, **`tools=`** — IDE hints stay faithful.


In [ ]:
def invoke(*, model: str, temperature: float = 0.2, stream: bool = False) -> str:
    mode = "sse" if stream else "batch"
    return f"{model}|{temperature}|{mode}"


print(invoke(model="gpt-mini", stream=True))


## 3 · Combined staging

*Explanation:* **`prompt` positional-only + hyperparameters keyword-only** mirrors SDK ergonomics.


In [ ]:
def chat(prompt: str, /, *, model: str, max_tokens: int = 512) -> str:
    return f"{model}({max_tokens}) <- {prompt.split()[0]}"


print(chat("Explain asyncio queues", model="opus"))


---

## Progressive drills — **easy → harder**

**Signature hygiene** prevents accidental argument drift when merging LangChain-style wrappers.


### A · Easiest — positional clarity

Reserve positional slots for payloads only.


In [ ]:
def embed(text: str, /, dims: int = 768) -> tuple[str, int]:
    return ("fake", dims)


print(embed("chunk body", dims=512))


### B · Medium — forbid ambiguous booleans

Keyword-only removes **`True`** positional traps.


In [ ]:
def rerank(query: str, /, *, aggressive: bool = False) -> str:
    return f"{query}:{int(aggressive)}"


print(rerank("latency regressions", aggressive=True))


### C · Harder — façade forwarding

Compose vendors without leaking positional quirks upward.


In [ ]:
def vendor_raw(blob: str, /, vendor_tag: str) -> str:
    return f"[{vendor_tag}]{blob}"


def façade(user_prompt: str, /, *, vendor_tag: str, polish: bool = False) -> str:
    core = vendor_raw(user_prompt.strip(), vendor_tag)
    return core.upper() if polish else core


print(façade(" hello ", vendor_tag="az", polish=True))


### Exercise — `tool_call`

Implement **`def tool_call(name: str, /, *, payload: dict[str, str], dry_run: bool = False) -> str`** returning **`NAME|dry=<bool>|<sorted kv tuples>`** using **`repr(tuple(sorted(payload.items())))`** for the tail.

Example: **`tool_call("search", payload={"q": "rag"}, dry_run=True)`** starts with **`SEARCH|dry=True|`**.


In [ ]:
def tool_call(name: str, /, *, payload: dict[str, str], dry_run: bool = False) -> str:
    raise NotImplementedError


sample = tool_call("search", payload={"q": "rag"}, dry_run=True)
assert sample.startswith("SEARCH|dry=True|")
assert "'q'" in sample
print("OK")


### Solution — tool_call

<details>
<summary>Click to expand</summary>

```python
def tool_call(name: str, /, *, payload: dict[str, str], dry_run: bool = False) -> str:
    tail = repr(tuple(sorted(payload.items())))
    return f"{name.upper()}|dry={dry_run}|{tail}"
```

</details>
